<a href="https://colab.research.google.com/github/aburkov/theDRLbook/blob/main/gradient_ascent_pytorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Gradient Ascent in PyTorch

This notebook solves the same optimization problem as the NumPy version,

$$J(\theta_1, \theta_2) = 10 - (\theta_1 - 2)^2 - (\theta_2 - 3)^2,$$

but instead of approximating the gradient with finite differences, it uses PyTorch's **automatic differentiation** to compute the gradient exactly.

## Setup

`torch` is preinstalled on Colab, so the import below is all we need.

In [1]:
import torch

## The objective function

Almost identical to the NumPy version, except `theta` is a tensor. We index into it (`theta[0]`, `theta[1]`) so the two components stay linked to `theta` in the computation graph, and the returned value carries the record of how it was built.

In [2]:
def objective_function(theta):
    theta1, theta2 = theta[0], theta[1]
    return 10 - (theta1 - 2)**2 - (theta2 - 3)**2

## Computing the gradient with autograd

Instead of finite differences, we evaluate the objective and call `.backward()`. PyTorch *accumulates* gradients into `.grad`, so we zero any previous gradient first (guarding against the first call, when `.grad` is still `None`).

In [3]:
def compute_gradient(obj, theta):
    if theta.grad is not None:
        theta.grad.zero_()

    value = obj(theta)
    value.backward()

    return theta.grad

## The gradient ascent loop

The update itself is part of our optimization procedure, not part of the function we are differentiating, so we wrap it in `torch.no_grad()` to keep it out of the computation graph. This also avoids an error from modifying a tracked tensor in place.

In [4]:
def gradient_ascent(theta, learning_rate, num_iterations, obj):
    for _ in range(num_iterations):
        grad = compute_gradient(obj, theta)

        with torch.no_grad():
            theta += learning_rate * grad

    return theta

## Run it

The one structural difference from NumPy is that `theta_start` is created with `requires_grad=True` so autograd tracks it from the start. We convert tensors back to plain numbers for printing: `.detach()` separates a tensor from its graph before converting to a NumPy array, and `.item()` extracts a single scalar.

In [5]:
theta_start = torch.tensor([3.8, 1.5], requires_grad=True)
learning_rate = 0.1
max_iterations = 30

theta_optimal = gradient_ascent(
    theta_start,
    learning_rate,
    max_iterations,
    objective_function
)

with torch.no_grad():
    optimal_value = objective_function(theta_optimal)

print(f"Optimal theta: {theta_optimal.detach().numpy()}")
print(f"Objective function value at optimal theta: {optimal_value.item()}")

Optimal theta: [2.0022283 2.9981432]
Objective function value at optimal theta: 9.999991416931152


Expected output:

```
Optimal theta: [2.0022283 2.9981432]
Objective function value at optimal theta: 9.999991416931152
```

This matches the NumPy result to about six significant figures. The tiny difference is because PyTorch tensors default to 32-bit floats (`float32`) while NumPy used 64-bit (`float64`). The key point is that PyTorch computed all partial derivatives exactly in a single backward pass, rather than approximating them.